# Team 11 · Checkpoint 3: Bronze and Silver layers
**Project:** Road Accident Risk Pipeline for usage-based auto insurance  
**Client brief:** Accenture, Data Engineer (Hyderabad)  
**Dataset:** US Accidents (2016–2023), Sobhan Moosavi et al., Kaggle. 7,728,394 rows, 46 columns. Licence CC BY-NC-SA 4.0. Cite: Moosavi et al. (2019).

**What this notebook does**
1. **Bronze:** loads the raw CSV as-is into a Delta table and adds load metadata.
2. **Silver:** cleans the data with PySpark: fixes data types, parses dates, adds time features, groups weather labels, removes duplicates and separates invalid rows.
3. **Quality checks:** measures row counts, null rates and rule breaks, and saves them in a log table.

**Gaps this closes (from our Checkpoint 1 matrix):** Spark/PySpark on large data (4.33), lakehouse layers (3.67), pipelines and ETL (3.33), data quality (2.67), cloud platform (3.00).

**How to run:** run the cells from top to bottom. Leave `SAMPLE_FRACTION = 0.01` (1% sample) while testing. When everything works, set it to `1.0` and run once on the full data.

**AI use:** the first draft of this code was written with Claude (an AI assistant). Each team member has read and run it, and can explain every step. This is recorded in our learning log.

## 0. Setup
*Lead: Sandeep*

These settings say where the raw file is and where our tables go. `SAMPLE_FRACTION` controls how much of the data the silver step uses, so we don't use up the daily compute quota while testing.

In [0]:
from pyspark.sql import functions as F

CATALOG = "workspace"
SCHEMA  = "accidents"
RAW_PATH = "/Volumes/workspace/accidents/raw_files/US_Accidents_March23.csv"

SAMPLE_FRACTION = 1.0   # 0.01 = 1% sample for testing; set to 1.0 for the final full run
EXPECTED_ROWS = 7728394  # row count published for this dataset

def tbl(name):
    # Full table name, e.g. workspace.accidents.bronze_accidents
    return f"{CATALOG}.{SCHEMA}.{name}"

print("Raw file :", RAW_PATH)
print("Tables in:", f"{CATALOG}.{SCHEMA}")
print("Sample   :", SAMPLE_FRACTION)

Raw file : /Volumes/workspace/accidents/raw_files/US_Accidents_March23.csv
Tables in: workspace.accidents
Sample   : 1.0


## 1. Bronze layer: load the raw data as-is
*Lead: Sandeep*

**Rule for bronze:** keep the data exactly as it arrived, so we can always go back to the original.
- Every column is read as **text (string)**. We do not change any values here.
- We add two **metadata columns**: `_ingest_ts` (when we loaded it) and `_source_file` (which file it came from).
- **One small change:** Delta tables do not allow some characters in column names, such as `(`, `)` and `%`. So `Distance(mi)` becomes `Distance_mi`, and `Humidity(%)` becomes `Humidity_pct`. The values are untouched.

In [0]:
import re

raw = (spark.read
       .option("header", True)
       .option("inferSchema", False)   # keep everything as text in bronze
       .csv(RAW_PATH))

def delta_safe(name):
    name = name.replace("(%)", "_pct").replace("(", "_").replace(")", "")
    return re.sub(r"[ ,;{}\n\t=%]", "_", name)

old_names = raw.columns
new_names = [delta_safe(c) for c in old_names]
renamed = {o: n for o, n in zip(old_names, new_names) if o != n}
print("Columns renamed for Delta:", renamed)

bronze = (raw.toDF(*new_names)
          .withColumn("_ingest_ts", F.current_timestamp())
          .withColumn("_source_file", F.col("_metadata.file_path")))

(bronze.write
       .mode("overwrite")
       .option("overwriteSchema", "true")
       .saveAsTable(tbl("bronze_accidents")))

print("Bronze table written:", tbl("bronze_accidents"))

Columns renamed for Delta: {'Distance(mi)': 'Distance_mi', 'Temperature(F)': 'Temperature_F', 'Wind_Chill(F)': 'Wind_Chill_F', 'Humidity(%)': 'Humidity_pct', 'Pressure(in)': 'Pressure_in', 'Visibility(mi)': 'Visibility_mi', 'Wind_Speed(mph)': 'Wind_Speed_mph', 'Precipitation(in)': 'Precipitation_in'}
Bronze table written: workspace.accidents.bronze_accidents


In [0]:
bronze = spark.table(tbl("bronze_accidents"))
bronze_rows = bronze.count()
print(f"Bronze rows   : {bronze_rows:,}")
print(f"Expected rows : {EXPECTED_ROWS:,}")
print("Match:", bronze_rows == EXPECTED_ROWS)
display(bronze.limit(5))

Bronze rows   : 7,728,394
Expected rows : 7,728,394
Match: True


ID,Source,Severity,Start_Time,End_Time,Start_Lat,Start_Lng,End_Lat,End_Lng,Distance_mi,Description,Street,City,County,State,Zipcode,Country,Timezone,Airport_Code,Weather_Timestamp,Temperature_F,Wind_Chill_F,Humidity_pct,Pressure_in,Visibility_mi,Wind_Direction,Wind_Speed_mph,Precipitation_in,Weather_Condition,Amenity,Bump,Crossing,Give_Way,Junction,No_Exit,Railway,Roundabout,Station,Stop,Traffic_Calming,Traffic_Signal,Turning_Loop,Sunrise_Sunset,Civil_Twilight,Nautical_Twilight,Astronomical_Twilight,_ingest_ts,_source_file
A-7496677,Source1,3,2018-11-06 08:32:26,2018-11-06 09:01:47,40.67122,-73.514872,40.67146,-73.513549,0.071,At Wantagh Pkwy - Accident.,Sunrise Hwy,Wantagh,Nassau,NY,11793,US,US/Eastern,KFRG,2018-11-06 08:30:00,52.0,null,100.0,29.97,0.8,ESE,6.9,null,Overcast,False,False,False,False,False,False,False,False,False,False,False,False,False,Day,Day,Day,Day,2026-11-07T05:50:18.192Z,dbfs:/Volumes/workspace/accidents/raw_files/US_Accidents_March23.csv
A-7496678,Source1,4,2018-11-06 08:35:15,2018-11-06 09:05:11,40.543429,-79.82364,40.60851,-79.94556,7.82,Closed between Freeport Rd/Exit 5 and PA-8/Butler Vly Interchange/Exit 39 - Road closed due to accident.,I-76 W,Pittsburgh,Allegheny,PA,15238,US,US/Eastern,KAGC,2018-11-06 08:25:00,53.1,null,100.0,29.69,6.0,SSE,13.8,0.03,Light Rain,False,False,False,False,False,False,False,False,False,False,False,False,False,Day,Day,Day,Day,2026-11-07T05:50:18.192Z,dbfs:/Volumes/workspace/accidents/raw_files/US_Accidents_March23.csv
A-7496679,Source1,2,2018-11-06 08:20:26,2018-11-06 08:50:20,40.427575,-74.438578,40.401558,-74.45264,1.944,Between RT-18/Exit 9 and CR-612/Exit 8 - Accident. Right lane blocked. Vehicle spun around.,New Jersey Tpke S,East Brunswick,Middlesex,NJ,08816,US,US/Eastern,KLDJ,2018-11-06 08:15:00,51.6,null,95.0,29.97,1.0,Calm,null,null,Overcast,False,False,False,False,False,False,False,False,False,False,False,False,False,Day,Day,Day,Day,2026-11-07T05:50:18.192Z,dbfs:/Volumes/workspace/accidents/raw_files/US_Accidents_March23.csv
A-7496680,Source1,2,2018-11-06 08:26:25,2018-11-06 08:56:05,40.464992,-74.289103,40.4413,-74.26371999999998,2.112,At Cheesequake Park Rd - Accident. Right lane blocked.,Garden State Pkwy S,South Amboy,Middlesex,NJ,08879,US,US/Eastern,KLDJ,2018-11-06 08:35:00,52.0,null,96.0,29.95,1.0,Calm,null,null,Overcast,False,False,False,False,False,False,False,False,False,False,False,False,False,Day,Day,Day,Day,2026-11-07T05:50:18.192Z,dbfs:/Volumes/workspace/accidents/raw_files/US_Accidents_March23.csv
A-7496681,Source1,3,2018-11-06 08:32:26,2018-11-06 09:01:47,40.67157,-73.51365899999998,40.671145,-73.51592099999998,0.122,At Wantagh Pkwy - Accident.,Sunrise Hwy,Wantagh,Nassau,NY,11793,US,US/Eastern,KFRG,2018-11-06 08:30:00,52.0,null,100.0,29.97,0.8,ESE,6.9,null,Overcast,False,False,False,False,False,False,False,False,False,False,False,False,False,Day,Day,Day,Day,2026-11-07T05:50:18.192Z,dbfs:/Volumes/workspace/accidents/raw_files/US_Accidents_March23.csv


## 2. Silver layer: clean and standardise
*Leads: Roshith, Prashansa, Robin*

Silver turns the raw text into clean, typed, analysis-ready data. Steps:
1. **Take the working set:** the 1% sample while testing, the full data for the final run.
2. **Fix data types:** numbers become numbers, True/False become booleans. We use `try_cast`, so a bad value becomes empty (null) instead of crashing the job.
3. **Parse dates:** some times are stored as `2016-02-08 05:46:00` and others with extra fractions of a second, like `2016-02-08 05:46:00.000000000`. We keep the first 19 characters, so both formats parse the same way.
4. **Add time features:** year, month, weekday, hour, and accident duration in minutes.
5. **Group weather labels:** the raw data has over 100 weather descriptions. We group them into 8 categories.
6. **Remove duplicate IDs** and **separate invalid rows** (with a reason) into a rejects table instead of silently deleting them.

In [0]:
# Step 1: working set
# Save the sample once as a table, so every later step uses exactly the same rows.
# (On serverless, sample() is re-run on every count and can pick different rows each time.)
src = spark.table(tbl("bronze_accidents"))
if SAMPLE_FRACTION < 1.0:
    (src.sample(fraction=SAMPLE_FRACTION, seed=42)
        .write.mode("overwrite").option("overwriteSchema", "true")
        .saveAsTable(tbl("silver_input_sample")))
    src = spark.table(tbl("silver_input_sample"))
src_rows = src.count()
print(f"Rows in this run: {src_rows:,}")

Rows in this run: 7,728,394


In [0]:
# Steps 2 and 3: data types and dates
def to_num(col, kind="double"):
    return F.expr(f"try_cast(`{col}` AS {kind})")

def to_ts(col):
    # keep 'yyyy-MM-dd HH:mm:ss' (first 19 characters); extra fractions of a second are dropped
    return F.expr(f"try_to_timestamp(substring(`{col}`, 1, 19), 'yyyy-MM-dd HH:mm:ss')")

road_features = ["Amenity", "Bump", "Crossing", "Give_Way", "Junction", "No_Exit", "Railway",
                 "Roundabout", "Station", "Stop", "Traffic_Calming", "Traffic_Signal", "Turning_Loop"]
numeric_cols = ["Start_Lat", "Start_Lng", "Distance_mi", "Temperature_F", "Wind_Chill_F",
                "Humidity_pct", "Pressure_in", "Visibility_mi", "Wind_Speed_mph", "Precipitation_in"]

typed = src.select(
    F.col("ID"),
    F.col("Source"),
    to_num("Severity", "int").alias("Severity"),
    F.col("Start_Time").alias("start_time_raw"),
    to_ts("Start_Time").alias("start_time"),
    to_ts("End_Time").alias("end_time"),
    *[to_num(c).alias(c) for c in numeric_cols],
    F.trim("City").alias("City"),
    F.trim("County").alias("County"),
    F.upper(F.trim("State")).alias("State"),
    F.col("Zipcode"),
    F.col("Timezone"),
    F.trim("Weather_Condition").alias("Weather_Condition"),
    *[to_num(c, "boolean").alias(c) for c in road_features],
    F.col("Sunrise_Sunset"),
    F.col("_ingest_ts"),
)
typed.printSchema()

root
 |-- ID: string (nullable = true)
 |-- Source: string (nullable = true)
 |-- Severity: integer (nullable = true)
 |-- start_time_raw: string (nullable = true)
 |-- start_time: timestamp (nullable = true)
 |-- end_time: timestamp (nullable = true)
 |-- Start_Lat: double (nullable = true)
 |-- Start_Lng: double (nullable = true)
 |-- Distance_mi: double (nullable = true)
 |-- Temperature_F: double (nullable = true)
 |-- Wind_Chill_F: double (nullable = true)
 |-- Humidity_pct: double (nullable = true)
 |-- Pressure_in: double (nullable = true)
 |-- Visibility_mi: double (nullable = true)
 |-- Wind_Speed_mph: double (nullable = true)
 |-- Precipitation_in: double (nullable = true)
 |-- City: string (nullable = true)
 |-- County: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Zipcode: string (nullable = true)
 |-- Timezone: string (nullable = true)
 |-- Weather_Condition: string (nullable = true)
 |-- Amenity: boolean (nullable = true)
 |-- Bump: boolean (nullable 

In [0]:
# Step 4: time features and accident duration
timed = (typed
    .withColumn("year",        F.year("start_time"))
    .withColumn("month",       F.month("start_time"))
    .withColumn("weekday_num", F.dayofweek("start_time"))        # 1 = Sunday ... 7 = Saturday
    .withColumn("weekday",     F.date_format("start_time", "E")) # Sun, Mon, ...
    .withColumn("hour",        F.hour("start_time"))
    .withColumn("duration_min",
                F.round((F.unix_timestamp("end_time") - F.unix_timestamp("start_time")) / 60, 1))
    .withColumn("is_severe",   F.col("Severity") >= 3))         # severity 3-4 = severe
display(timed.select("ID", "start_time_raw", "start_time", "end_time", "duration_min",
                     "year", "month", "weekday", "hour", "Severity", "is_severe").limit(5))

ID,start_time_raw,start_time,end_time,duration_min,year,month,weekday,hour,Severity,is_severe
A-1079659,2021-04-16 06:52:26,2021-04-16T06:52:26.000Z,2021-04-16T07:45:00.000Z,52.6,2021,4,Fri,6,2,false
A-1079660,2021-04-16 06:55:58,2021-04-16T06:55:58.000Z,2021-04-16T07:55:46.000Z,59.8,2021,4,Fri,6,2,false
A-1079661,2021-04-16 07:06:53,2021-04-16T07:06:53.000Z,2021-04-16T07:57:39.000Z,50.8,2021,4,Fri,7,2,false
A-1079662,2021-04-16 07:09:30,2021-04-16T07:09:30.000Z,2021-04-16T08:27:53.000Z,78.4,2021,4,Fri,7,2,false
A-1079663,2021-04-16 07:40:53,2021-04-16T07:40:53.000Z,2021-04-16T08:47:00.000Z,66.1,2021,4,Fri,7,2,false


In [0]:
# Step 5: group the 100+ weather labels into 8 categories
w = F.lower(F.col("Weather_Condition"))
weather_category = (
    F.when(w.isNull() | (w == ""), "Unknown")
     .when(w.rlike("thunder|t-storm|tornado|funnel|squall"), "Thunderstorm")
     .when(w.rlike("snow|sleet|ice|freezing|hail|wintry"), "Snow/Ice")
     .when(w.rlike("rain|drizzle|shower"), "Rain")
     .when(w.rlike("fog|mist|haze|smoke|dust|sand|ash"), "Fog/Haze/Dust")
     .when(w.rlike("cloud|overcast"), "Cloudy")
     .when(w.rlike("clear|fair"), "Clear")
     .otherwise("Other"))
weathered = timed.withColumn("weather_category", weather_category)

print("Distinct raw weather labels in this run:", weathered.select("Weather_Condition").distinct().count())
display(weathered.groupBy("weather_category").count().orderBy(F.desc("count")))

Distinct raw weather labels in this run: 145


weather_category,count
Clear,3405216
Cloudy,3163728
Rain,512097
Fog/Haze/Dust,202167
Snow/Ice,179749
Unknown,173459
Thunderstorm,88726
Other,3252


In [0]:
# Check which raw labels landed in each category (review this list once by eye)
display(weathered.groupBy("weather_category", "Weather_Condition").count()
        .orderBy("weather_category", F.desc("count")))

weather_category,Weather_Condition,count
Clear,Fair,2560802
Clear,Clear,808743
Clear,Fair / Windy,35671
Cloudy,Mostly Cloudy,1016195
Cloudy,Cloudy,817082
Cloudy,Partly Cloudy,698972
Cloudy,Overcast,382866
Cloudy,Scattered Clouds,204829
Cloudy,Cloudy / Windy,17035
Cloudy,Mostly Cloudy / Windy,16508


In [0]:
# Step 6: remove duplicate IDs, then separate invalid rows with a reason
deduped = weathered.dropDuplicates(["ID"])
dedup_rows = deduped.count()
duplicates_removed = src_rows - dedup_rows

reject_reason = (
    F.when(F.col("start_time").isNull(), "unparseable start time")
     .when(F.col("Severity").isNull() | ~F.col("Severity").between(1, 4), "severity outside 1-4")
     .when(F.col("end_time") < F.col("start_time"), "end time before start time")
     .when(~F.col("Start_Lat").between(24, 50) | ~F.col("Start_Lng").between(-125, -66),
           "coordinates outside the contiguous US"))
flagged = deduped.withColumn("reject_reason", reject_reason)

silver  = flagged.filter(F.col("reject_reason").isNull()).drop("reject_reason")
rejects = flagged.filter(F.col("reject_reason").isNotNull())

silver.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(tbl("silver_accidents"))
rejects.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(tbl("silver_rejects"))

silver_rows  = spark.table(tbl("silver_accidents")).count()
rejected_rows = spark.table(tbl("silver_rejects")).count()
print(f"Rows in            : {src_rows:,}")
print(f"Duplicates removed : {duplicates_removed:,}")
print(f"Rejected rows      : {rejected_rows:,}")
print(f"Silver rows        : {silver_rows:,}")
print("Rows add up:", src_rows == duplicates_removed + rejected_rows + silver_rows)

Rows in            : 7,728,394
Duplicates removed : 0
Rejected rows      : 0
Silver rows        : 7,728,394
Rows add up: True


In [0]:
display(spark.table(tbl("silver_rejects")).groupBy("reject_reason").count().orderBy(F.desc("count")))
display(spark.table(tbl("silver_accidents")).limit(5))

reject_reason,count


ID,Source,Severity,start_time_raw,start_time,end_time,Start_Lat,Start_Lng,Distance_mi,Temperature_F,Wind_Chill_F,Humidity_pct,Pressure_in,Visibility_mi,Wind_Speed_mph,Precipitation_in,City,County,State,Zipcode,Timezone,Weather_Condition,Amenity,Bump,Crossing,Give_Way,Junction,No_Exit,Railway,Roundabout,Station,Stop,Traffic_Calming,Traffic_Signal,Turning_Loop,Sunrise_Sunset,_ingest_ts,year,month,weekday_num,weekday,hour,duration_min,is_severe,weather_category
A-1080323,Source2,3,2021-04-16 16:18:55,2021-04-16T16:18:55.000Z,2021-04-16T16:48:24.000Z,36.141048,-86.71929899999998,0.0,66.0,66.0,28.0,29.26,10.0,5.0,0.0,Nashville,Davidson,TN,37210,US/Central,Mostly Cloudy,false,false,false,false,false,false,false,false,false,false,false,false,false,Day,2026-11-07T05:50:18.192Z,2021,4,6,Fri,16,29.5,true,Cloudy
A-1081109,Source2,2,2021-04-15 12:28:57,2021-04-15T12:28:57.000Z,2021-04-15T15:08:16.000Z,36.06144000000001,-83.979568,0.0,62.0,62.0,41.0,28.98,10.0,7.0,0.0,Knoxville,Knox,TN,37938-4402,US/Eastern,Partly Cloudy,false,false,false,false,false,false,false,false,false,false,false,false,false,Day,2026-11-07T05:50:18.192Z,2021,4,5,Thu,12,159.3,false,Cloudy
A-1083137,Source2,3,2021-04-14 08:12:15,2021-04-14T08:12:15.000Z,2021-04-14T12:05:59.000Z,42.063564,-88.07755999999998,0.0,40.0,34.0,53.0,29.44,10.0,9.0,0.0,Schaumburg,Cook,IL,60195,US/Central,Fair,false,false,false,false,false,false,false,false,false,false,false,false,false,Day,2026-11-07T05:50:18.192Z,2021,4,4,Wed,8,233.7,true,Clear
A-1083181,Source2,4,2021-04-14 08:39:53,2021-04-14T08:39:53.000Z,2021-04-14T09:41:30.000Z,29.365740000000002,-98.519623,0.3400000035762787,74.0,74.0,81.0,29.23,7.0,3.0,0.0,San Antonio,Bexar,TX,78211-4300,US/Central,Cloudy,false,false,false,false,false,false,false,false,false,false,false,false,false,Day,2026-11-07T05:50:18.192Z,2021,4,4,Wed,8,61.6,true,Cloudy
A-1083960,Source2,2,2021-04-13 16:11:17,2021-04-13T16:11:17.000Z,2021-04-13T17:16:44.000Z,41.424332,-81.645248,0.0,57.0,57.0,57.0,29.44,10.0,6.0,0.0,Cleveland,Cuyahoga,OH,44125,US/Eastern,Fair,false,false,false,false,false,false,false,false,false,false,false,false,false,Day,2026-11-07T05:50:18.192Z,2021,4,3,Tue,16,65.5,false,Clear


## 3. Quality checks and log table
*Leads: Chrislynn, Roshith*

Every run adds its results to a log table, `dq_log`, so we have a record of data quality over time. Each check has a **status**:
- **PASS:** the rule holds.
- **INFO:** a measurement to report (such as a null rate), not a failure.
- **WARN:** something to explain in our progress note.

**Note on missing weather values:** we do **not** fill them with guesses. Inventing temperatures or rainfall would make the analysis look more certain than it is. We keep them as null and report the null rates here.

In [0]:
from datetime import datetime

run_ts = datetime.now()
silver_df = spark.table(tbl("silver_accidents"))
checks = []

def log(layer, check, value, status, note=""):
    checks.append((run_ts, float(SAMPLE_FRACTION), layer, check, float(value), status, note))

# Row counts
log("bronze", "row_count", bronze_rows, "PASS" if bronze_rows == EXPECTED_ROWS else "WARN",
    f"expected {EXPECTED_ROWS}")
log("silver", "rows_in", src_rows, "INFO", "rows taken from bronze in this run")
log("silver", "duplicate_ids_removed", duplicates_removed, "PASS" if duplicates_removed == 0 else "WARN")
log("silver", "rows_rejected", rejected_rows, "INFO", "see silver_rejects for reasons")
log("silver", "rows_out", silver_rows, "INFO")
log("silver", "rows_reconcile",
    1 if src_rows == duplicates_removed + rejected_rows + silver_rows else 0,
    "PASS" if src_rows == duplicates_removed + rejected_rows + silver_rows else "WARN",
    "rows in = duplicates + rejects + silver")

# Null rates in key weather fields (percent of silver rows)
weather_fields = ["Temperature_F", "Humidity_pct", "Visibility_mi", "Wind_Speed_mph",
                  "Precipitation_in", "Weather_Condition"]
null_pct = silver_df.select([
    (F.avg(F.col(c).isNull().cast("int")) * 100).alias(c) for c in weather_fields
]).first().asDict()
for c in weather_fields:
    log("silver", f"null_pct_{c}", round(null_pct[c] or 0, 2), "INFO")

# Rules that must hold in silver
rules = silver_df.select(
    F.sum((~F.col("Severity").between(1, 4)).cast("int")).alias("severity_outside_1_4"),
    F.sum((F.col("end_time") < F.col("start_time")).cast("int")).alias("end_before_start"),
    F.sum(F.col("start_time").isNull().cast("int")).alias("null_start_time"),
    (F.count("*") - F.countDistinct("ID")).alias("duplicate_ids"),
).first().asDict()
for name, value in rules.items():
    value = value or 0
    log("silver", name, value, "PASS" if value == 0 else "WARN")

# Share of rows with unknown weather category
unknown_pct = silver_df.select(
    (F.avg((F.col("weather_category") == "Unknown").cast("int")) * 100).alias("p")).first()["p"] or 0
log("silver", "pct_weather_unknown", round(unknown_pct, 2), "INFO")

dq = spark.createDataFrame(checks,
    "run_ts timestamp, sample_fraction double, layer string, check_name string, value double, status string, note string")
dq.write.mode("append").saveAsTable(tbl("dq_log"))
display(dq)

run_ts,sample_fraction,layer,check_name,value,status,note
2026-11-07T06:03:45.862Z,1.0,bronze,row_count,7728394.0,PASS,expected 7728394
2026-11-07T06:03:45.862Z,1.0,silver,rows_in,7728394.0,INFO,rows taken from bronze in this run
2026-11-07T06:03:45.862Z,1.0,silver,duplicate_ids_removed,0.0,PASS,
2026-11-07T06:03:45.862Z,1.0,silver,rows_rejected,0.0,INFO,see silver_rejects for reasons
2026-11-07T06:03:45.862Z,1.0,silver,rows_out,7728394.0,INFO,
2026-11-07T06:03:45.862Z,1.0,silver,rows_reconcile,1.0,PASS,rows in = duplicates + rejects + silver
2026-11-07T06:03:45.862Z,1.0,silver,null_pct_Temperature_F,2.12,INFO,
2026-11-07T06:03:45.862Z,1.0,silver,null_pct_Humidity_pct,2.25,INFO,
2026-11-07T06:03:45.862Z,1.0,silver,null_pct_Visibility_mi,2.29,INFO,
2026-11-07T06:03:45.862Z,1.0,silver,null_pct_Wind_Speed_mph,7.39,INFO,


## 4. Summary for the progress note
Run this cell after the **full run** (`SAMPLE_FRACTION = 1.0`). It prints the numbers for our five-line progress note.

In [0]:
print(f"Bronze rows loaded      : {bronze_rows:,}")
print(f"Rows processed in silver: {src_rows:,}  (sample fraction {SAMPLE_FRACTION})")
print(f"Duplicate IDs removed   : {duplicates_removed:,}")
print(f"Rows rejected           : {rejected_rows:,}")
print(f"Silver rows             : {silver_rows:,}")
for c in weather_fields:
    print(f"Null % {c:<18}: {null_pct[c] or 0:.2f}%")
print(f"Weather 'Unknown' share : {unknown_pct:.2f}%")

Bronze rows loaded      : 7,728,394
Rows processed in silver: 7,728,394  (sample fraction 1.0)
Duplicate IDs removed   : 0
Rows rejected           : 0
Silver rows             : 7,728,394
Null % Temperature_F     : 2.12%
Null % Humidity_pct      : 2.25%
Null % Visibility_mi     : 2.29%
Null % Wind_Speed_mph    : 7.39%
Null % Precipitation_in  : 28.51%
Null % Weather_Condition : 2.24%
Weather 'Unknown' share : 2.24%
